<a href="https://colab.research.google.com/github/iordye/chatterbox-finetuning-with-mlflow/blob/main/Chatterbox_Ha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install huggingface_hub datasets librosa soundfile mlflow pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "naijavoices/naijavoices-dataset"

all_files = api.list_repo_files(repo_id, repo_type="dataset")
hausa_files = sorted([f for f in all_files if "hausa-batch-0" in f and f.endswith(".parquet")])

print(len(hausa_files))
hausa_files[:5]  # sanity check the naming/order

104


['hausa-batch-0/train-00000-of-00104.parquet',
 'hausa-batch-0/train-00001-of-00104.parquet',
 'hausa-batch-0/train-00002-of-00104.parquet',
 'hausa-batch-0/train-00003-of-00104.parquet',
 'hausa-batch-0/train-00004-of-00104.parquet']

In [ ]:
# Take first 25 shards, build hf:// paths (no manual download needed)
subset_files = hausa_files[:1]
data_files = [f"hf://datasets/{repo_id}/{f}" for f in subset_files]



In [ ]:
from datasets import load_dataset

naija_ha = load_dataset("parquet", data_files=data_files, split="train")

hausa-batch-0/train-00000-of-00104.parqu(…): reconstructing file:   0%|          |  0.00B /  477MB            

hausa-batch-0/train-00000-of-00104.parqu(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# Ensure the audio column is decoded
# (assuming the column name is 'audio')

def calculate_total_hours(dataset):
    # Calculate duration for each sample in seconds
    # len(x["array"]) gives the number of audio frames
    durations = [len(x["audio"]["array"]) / x["audio"]["sampling_rate"] for x in dataset]

    total_seconds = sum(durations)
    total_hours = total_seconds / 3600

    print(f"Total duration: {total_hours:.2f} hours")
    print(f"Total samples: {len(dataset)}")
    print(f"Average sample length: {total_seconds / len(dataset):.2f} seconds")

    return durations

#durations = calculate_total_hours(naija_ha)


In [ ]:
def is_valid(example):
    text = example["text"]
    audio = example["audio"]

    if text is None or not str(text).strip():
        return False

    if audio is None:
        return False

    duration = len(audio["array"]) / audio["sampling_rate"]

    return 3 <= duration <= 10

In [ ]:
filtered = naija_ha.filter(is_valid)

print("Original:", len(naija_ha))
print("Filtered:", len(filtered))

Filter:   0%|          | 0/2272 [00:00<?, ? examples/s]

Original: 2272
Filtered: 992


In [ ]:
duration_filtered = calculate_total_hours(filtered)

In [ ]:
!ffmpeg -version

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --enable-

In [ ]:
!unzip /content/chatterbox-finetuning.zip

In [ ]:
!git clone https://github.com/gokhaneraslan/chatterbox-finetuning.git
%cd chatterbox-finetuning
!pip install -r requirements.txt -q

Cloning into 'chatterbox-finetuning'...
remote: Enumerating objects: 281, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 281 (delta 91), reused 62 (delta 62), pack-reused 168 (from 1)
Receiving objects: 100% (281/281), 1.02 MiB | 22.62 MiB/s, done.
Resolving deltas: 100% (143/143), done.
/content/chatterbox-finetuning
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.3/93.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
filtered = filtered.select(range(100))

In [ ]:
!python setup.py

--- Chatterbox Pretrained Model Setup ---

Creating directory: pretrained_models
Mode: CHATTERBOX-TTS (Checking 5 files)
Downloading: ve.safetensors...
ve.safetensors: 100% 5.43M/5.43M [00:00<00:00, 21.0MiB/s]
Download complete: pretrained_models/ve.safetensors

Downloading: t3_cfg.safetensors...
t3_cfg.safetensors: 100% 1.98G/1.98G [00:30<00:00, 70.2MiB/s]
Download complete: pretrained_models/t3_cfg.safetensors

Downloading: s3gen.safetensors...
s3gen.safetensors: 100% 0.98G/0.98G [00:15<00:00, 66.3MiB/s]
Download complete: pretrained_models/s3gen.safetensors

Downloading: conds.pt...
conds.pt: 100% 105k/105k [00:00<00:00, 714kiB/s]  
Download complete: pretrained_models/conds.pt

Downloading: tokenizer.json...
tokenizer.json: 68.3kiB [00:00, 80.2MiB/s]
Download complete: pretrained_models/tokenizer.json


INSTALLATION COMPLETE (CHATTERBOX-TTS MOD)
All models are set up in 'pretrained_models/' folder.
Note: 'grapheme_mtl_merged_expanded_v1.json' was saved as 'tokenizer.json' for the n

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/chatterbox-finetuning/pretrained_models/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Vocabulary size:", tokenizer.get_vocab_size())

# Testing Audio Convert to WAV on one file

In [ ]:
from pathlib import Path
WAV_DIR = Path("/content/chatterbox-finetuning/MyTTSDataset/wavs")


In [ ]:
import soundfile as sf
import numpy as np

def save_audio(example, index):
    audio = example["audio"]

    array = audio["array"]
    sr = audio["sampling_rate"]

    filename = f"hausa_{index:07d}.wav"
    output_path = WAV_DIR / filename

    # Convert to numpy
    array = np.asarray(array)

    # Make mono if necessary
    if array.ndim > 1:
        array = np.mean(array, axis=0)

    sf.write(
        output_path,
        array,
        sr,
        subtype="PCM_16"
    )

    return filename

In [ ]:
filtered

Dataset({
    features: ['audio', 'speaker_id', 'text', 'language', 'gender', 'age_range', 'phase'],
    num_rows: 992
})

In [ ]:
example = filtered[1]

audio = '/mnt/SDD1/Hausa_Dataset_24k/hau_f_031_ag_spontaneous_003.flac'

array = np.asarray(audio["array"])

if array.ndim > 1:
    array = np.mean(array, axis=0)

test_path = Path(WAV_DIR) / "test.wav"

sf.write(
    test_path,
    array,
    audio["sampling_rate"],
    subtype="PCM_16"
)

print(test_path)

/content/chatterbox-finetuning/MyTTSDataset/wavs/test.wav


# Create metadata.csv file

In [ ]:
import csv
DATASET_DIR = Path("/content/chatterbox-finetuning/MyTTSDataset")
metadata_path = DATASET_DIR / "metadata.csv"

rows = []

for i, example in enumerate(filtered):

    filename = save_audio(example, i)

    text = str(example["text"]).strip()

    rows.append([
        Path(filename).stem,
        text,
        text
    ])

with open(
    metadata_path,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f,
        delimiter="|",
        lineterminator="\n"
    )

    writer.writerows(rows)

print(metadata_path)
print("Rows:", len(rows))

/content/chatterbox-finetuning/MyTTSDataset/metadata.csv
Rows: 100


# Verify the metadata

In [ ]:
import pandas as pd

df = pd.read_csv(
    metadata_path,
    sep="|",
    header=None,
    names=[
        "filename",
        "raw_text",
        "normalized_text"
    ]
)

df.head()

,filename,raw_text,normalized_text
0,hausa_0000000,Na san shugaban ɗalibai maza na makarantar.,Na san shugaban ɗalibai maza na makarantar.
1,hausa_0000001,Matasa a kwanakin nan suna yin soyayya ta yana...,Matasa a kwanakin nan suna yin soyayya ta yana...
2,hausa_0000002,A cikin mutanen wa ya yiwa yarinyar fyade?,A cikin mutanen wa ya yiwa yarinyar fyade?
3,hausa_0000003,Wadanne irin gudummawar da maza za su iya baya...,Wadanne irin gudummawar da maza za su iya baya...
4,hausa_0000004,Wasu Yarbawa a Kudancin Najeriya sun yi imani ...,Wasu Yarbawa a Kudancin Najeriya sun yi imani ...


In [ ]:
df.shape

(100, 3)

# verify every WAV exists

In [ ]:
missing = []

for filename in df["filename"]:
    path = WAV_DIR / f"{filename}.wav"

    if not path.exists():
        missing.append(filename)

print("Missing audio:", len(missing))

if missing:
    print(missing[:10])

Missing audio: 0


# Confirming Audio Duration Again after conversion

In [ ]:
import librosa

durations = []

for filename in df["filename"]:

    path = WAV_DIR / f"{filename}.wav"

    duration = librosa.get_duration(
        filename=str(path)
    )

    durations.append(duration)

df["duration"] = durations

print(df["duration"].describe())

In [ ]:
from collections import Counter

chars = Counter()

for text in filtered["text"]:
    chars.update(text)

print("Unique characters:", len(chars))

for char, count in chars.most_common():
    print(repr(char), count)

Unique characters: 98
'a' 84347
' ' 65071
'n' 30151
'i' 28951
'u' 15457
'k' 14059
'r' 13431
's' 13231
'y' 11482
'd' 11102
'm' 10197
't' 9526
'e' 8847
'b' 7532
'w' 7166
'o' 6927
'h' 5603
'g' 5358
'l' 4915
'.' 4162
'c' 3958
'f' 3916
'j' 3152
'z' 2637
'A' 1632
'ɗ' 1585
'ƙ' 1318
"'" 1251
'M' 1019
',' 927
'K' 781
'S' 632
'N' 614
'B' 511
'Y' 464
'?' 446
'Z' 429
'I' 345
'W' 343
'D' 339
'-' 311
'T' 306
'G' 259
'H' 246
'ɓ' 239
'J' 237
'L' 205
'F' 172
'E' 159
'ƴ' 140
'p' 121
'R' 117
'C' 111
'"' 102
'!' 95
'Ɗ' 71
'O' 68
'Ƙ' 65
'’' 63
'U' 52
'P' 38
'Ƴ' 31
'v' 28
'q' 23
'2' 17
'\u200b' 16
'0' 15
'1' 12
'‘' 12
'ʼ' 11
'/' 10
'x' 9
'V' 8
'\xa0' 8
'5' 7
'ã' 7
'9' 7
'Ɓ' 5
'(' 5
')' 5
':' 5
'6' 4
'4' 4
'3' 3
'7' 3
';' 2
'8' 2
'à' 2
'í' 1
'ù' 1
'ú' 1
'_' 1
'ā' 1
'Q' 1
'“' 1
'”' 1
'ì' 1
'ò' 1


In [ ]:
vocab = tokenizer.get_vocab()

dataset_chars = set(chars.keys())

supported = sorted(
    dataset_chars.intersection(vocab.keys())
)

missing = sorted(
    dataset_chars - set(vocab.keys())
)

print("Characters in dataset:", len(dataset_chars))
print("Supported:", len(supported))
print("Missing:", len(missing))

print("\nMissing characters:")
for char in missing:
    print(repr(char))

Characters in dataset: 98
Supported: 98
Missing: 0

Missing characters:


In [ ]:
test_chars = ["ɓ", "ɗ", "ƙ", "ƴ"]

vocab = tokenizer.get_vocab()

for char in test_chars:
    if char in vocab:
        print(f"✓ {char!r} is supported → token ID: {vocab[char]}")
    else:
        print(f"✗ {char!r} is NOT supported")

✓ 'ɓ' is supported → token ID: 428
✓ 'ɗ' is supported → token ID: 432
✓ 'ƙ' is supported → token ID: 814
✓ 'ƴ' is supported → token ID: 838


# Begin Training

In [ ]:
!python train.py

2026-10-02 08:46:48.227596: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/perth/perth_net/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
2026/10/02 08:47:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/02 08:47:04 INFO mlflow.store.db.utils: Updating database tables
2026/10/02 08:47:06 INFO mlflow.tracking.fluent: Experiment with name 'Chatterbox-Finetuning' does not exist. Creating a new experiment.
2026-10-02 08:47:06 

In [ ]:
!zip -r mlruns.zip /content/chatterbox-finetuning/mlruns


  adding: content/chatterbox-finetuning/mlruns/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/84b37829f5534033bbce33b78c312fd5/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/84b37829f5534033bbce33b78c312fd5/artifacts/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/84b37829f5534033bbce33b78c312fd5/artifacts/checkpoints/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/84b37829f5534033bbce33b78c312fd5/artifacts/checkpoints/step-2/ (stored 0%)
  adding: content/chatterbox-finetuning/mlruns/1/84b37829f5534033bbce33b78c312fd5/artifacts/checkpoints/step-2/model.safetensors


zip error: Interrupted (aborting)


In [ ]:
!python inference.py

In [ ]:
import mlflow

In [ ]:
!pkill -f "mlflow ui" || true

^C


In [ ]:
import subprocess, time

mlflow_proc = subprocess.Popen(
    ["mlflow", "ui",
     "--backend-store-uri", "sqlite:////content/chatterbox-finetuning/mlflow.db",
     "--host", "0.0.0.0",
     "--port", "5002",
     "--allowed-hosts", "*",
     "--cors-allowed-origins", "*"],
    cwd="/content/chatterbox-finetuning",
    stdout=open("/content/mlflow_ui.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(8)
!curl -sI http://127.0.0.1:5002 | head -n 1
!tail -n 10 /content/mlflow_ui.log

Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled. Allowed hosts: *. CORS origins: *.
2026/10/02 09:52:02 INFO:     Uvicorn running on http://0.0.0.0:5002 (Press CTRL+C to quit)
2026/10/02 09:52:02 INFO:     Started parent process [24371]


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:5002 > cf.log 2>&1 &
import time; time.sleep(8)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' cf.log | head -1

https://power-licence-fireplace-memo.trycloudflare.com


In [ ]:
import sqlite3
con = sqlite3.connect("/content/chatterbox-finetuning/mlflow.db")
print(con.execute("select experiment_id, name from experiments").fetchall())
print(con.execute("select count(*) from runs").fetchall())

[(1, 'Chatterbox-Finetuning'), (0, 'Default')]
[(1,)]


In [ ]:
!pip show perth